<a href="https://colab.research.google.com/github/shaikhabdulwasay/Medical-QLoRA-Llama3/blob/main/medical_llama_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- 1. INSTALLATION ---
import os
import sys
import subprocess

def install_dependencies():
    print("Starting installation of dependencies...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"])
    packages = [
        "transformers", "trl", "peft", "accelerate", "bitsandbytes",
        "datasets", "huggingface_hub", "evaluate", "sentencepiece", "protobuf"
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps"] + packages)
    print("Dependencies installed.")

install_dependencies()

# --- 2. GPU & ENVIRONMENT SETUP ---
import unsloth # MUST BE IMPORTED FIRST
from unsloth import FastLanguageModel
import torch
from pynvml import *
import random
import numpy as np
import logging
from transformers import set_seed, AutoTokenizer
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback
from peft import PeftModel
import gc

def verify_gpu():
    if not torch.cuda.is_available():
        raise RuntimeError("❌ GPU not detected.")
    gpu_props = torch.cuda.get_device_properties(0)
    print(f"Found GPU: {gpu_props.name} | Total VRAM: {gpu_props.total_memory / (1024**3):.2f} GB")

verify_gpu()

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
SEED = 42
set_seed(SEED)

# --- 3. CONFIGURATION CONSTANTS ---
MODEL_NAME = "unsloth/llama-3-8b-instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
SYSTEM_PROMPT = "You are a helpful medical assistant. Answer the following medical question accurately and concisely based on clinical knowledge."

# --- 4. DATASET PREPARATION ---
import re

def clean_text(text):
    if not text: return ""
    text = re.sub(r'<.*?>', '', text)
    return text.strip()

def prepare_dataset():
    print("[STEP] Loading and Cleaning Dataset...")
    dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")
    df = dataset.to_pandas()
    df['input'] = df['input'].apply(clean_text)
    df['output'] = df['output'].apply(clean_text)
    df = df.dropna(subset=['input', 'output'])
    df = df[(df['input'] != "") & (df['output'] != "")]
    df = df[df['output'].str.len() >= 20]

    from datasets import Dataset
    cleaned_ds = Dataset.from_pandas(df)
    if 'instruction' not in cleaned_ds.column_names:
        cleaned_ds = cleaned_ds.add_column("instruction", [SYSTEM_PROMPT] * len(cleaned_ds))
    return cleaned_ds

# --- 5. MODEL & TRAINING SETUP ---
def load_and_train():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype = None,
        load_in_4bit = True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0.05,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = SEED,
    )

    dataset = prepare_dataset()

    def formatting_prompts_func(examples):
        texts = []
        for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
            messages = [
                {"role": "system", "content": inst},
                {"role": "user", "content": inp},
                {"role": "assistant", "content": out},
            ]
            texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
        return { "text" : texts }

    dataset = dataset.map(formatting_prompts_func, batched=True)
    split_ds = dataset.train_test_split(test_size=0.1, seed=SEED)

    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = split_ds['train'],
        eval_dataset = split_ds['test'],
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        args = TrainingArguments(
            output_dir = "./outputs",
            per_device_train_batch_size = 2,
            gradient_accumulation_steps = 4,
            num_train_epochs = 1,
            learning_rate = 2e-4,
            fp16 = not unsloth.is_bfloat16_supported(),
            bf16 = unsloth.is_bfloat16_supported(),
            logging_steps = 10,
            save_steps = 100,
            seed = SEED
        )
    )

    print("[STEP] Starting Training...")
    trainer.train()

    print("[STEP] Saving Adapters...")
    model.save_pretrained("outputs/medical_lora_adapter")
    tokenizer.save_pretrained("outputs/medical_lora_adapter")
    return model, tokenizer

model, tokenizer = load_and_train()

# --- 6. INFERENCE EVALUATION ---
print("[STEP] Reloading for Inference...")
del model
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)
model = PeftModel.from_pretrained(model, "outputs/medical_lora_adapter")
FastLanguageModel.for_inference(model)

test_qs = ["What are the symptoms of appendicitis?", "How is pneumonia diagnosed?"]
for q in test_qs:
    prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou are a medical assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n{q}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=128, use_cache=True)
    print(f"Q: {q}\nA: {tokenizer.batch_decode(out)[0].split('assistant<|end_header_id|>')[-1].strip()}\n")